In [13]:
import pandas as pd
import pathlib
import ollama
import sys

In [14]:
DATABASE_GOOD = "sqlite:///../data/track2_good.db"
DATABASE_BAD = "sqlite:///../data/track2_bad.db"
OUTPUT = '../data/output.txt'
PROMPTS = '../prompts/prompts.txt'
START_SEQUENCE = 10
END_SEQUENCE = 25

# Preprocessing


In [15]:
df = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    DATABASE_GOOD
)

def preprocess(df):
    df['timestamp'] = pd.to_datetime(df["timestamp"], utc=True).astype("int64") // 10**6
    df['timestamp'] = (df['timestamp'] - min(df['timestamp'])) / 1000

    df['position_x'] = df['position_x'].astype(int)
    df['position_y'] = df['position_y'].astype(int)
    df['position_z'] = df['position_z'].astype(int)

    df['acceleration_x'] = df['acceleration_x'].astype(int)
    df['acceleration_y'] = df['acceleration_y'].astype(int)
    df['acceleration_z'] = df['acceleration_z'].astype(int)
    df['yaw'] = df['yaw'].round(decimals=2)
    df['speed'] = (df['speed'] * 3.6).astype(int)
    
    return df

df_fast = preprocess(df)

df_fast = df_fast.iloc[::19, :]
print(df_fast.shape)

(51, 9)


In [16]:
df2 = pd.read_sql(
    """
    SELECT 
        timestamp_utc AS timestamp,
        acceleration_x,
        acceleration_y,
        acceleration_z,
        yaw,
        position_x,
        position_y,
        position_z,
        speed 
    FROM telemetry_samples
    WHERE distance_traveled != 0
    ORDER BY id
    """,
    DATABASE_BAD
)

df_slow = preprocess(df2)

x = max(df_slow["timestamp"]) / max(df_fast["timestamp"])

df_slow = df_slow.iloc[::int(19*x), :]
print(df_slow.shape)

(51, 9)


In [17]:
df_combined = df_fast.copy()
for col in df_fast.columns:
    df_combined[col] = list(zip(df_fast[col], df_slow[col]))

records = df_combined.to_dict(orient="records")
text = str(records)
for char in '[]{}()':
    text = text.replace(char, '')
    
text = text.replace('timestamp', '\ntimestamp')

with open(OUTPUT, "w", encoding="utf-8") as f:
    f.write(text)

In [20]:
with open(OUTPUT) as input_file:
    text = f'{input_file.readlines()[START_SEQUENCE:END_SEQUENCE]}'

text = text.strip('"')
text.replace('\'', '')

system_prompt = "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze. Timestamp ist immer in Sekunden. Jedes Attribut hat zwei Werte: Das zweite steht immer für die zu bewertende Runde, der erste Wert beschreibt eine optimale Runde die dir als Referenz dient. **Verwende dafür ausschließlich die Daten aus der Liste des Users**. Negative Prompt: Denk dir keine weiteren Daten aus, Bewerte nicht die ersten Werte der Attribute"
user_prompt = "Bewerte meine Fahrleistung, zeige mir klar die Unterschiede:"

resp = ollama.chat(
    model="nemotron-3-nano:30b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + f"\n```json\n{text}\n```"},
    ],
)

print(resp["message"]["content"])
pathlib.Path("prompts").mkdir(parents=True, exist_ok=True)
with open(PROMPTS, "a", encoding="utf-8") as file:
    file.writelines([
        f"Sequence: {START_SEQUENCE} - {END_SEQUENCE}\n",
        "System Prompt: " + str(system_prompt) + "\n",
        "User Prompt: " + str(user_prompt) + "\n",
        "Response: " + str(resp["message"]["content"]) + "\n\n",
    ])

Ab etwa 9 s erreicht du Lateralbeschleunigungen von bis +20 m/s², während die Referenzrunde nur 0‑5 m/s² vorsieht – das erzeugt starkes Unter‑/Übersteuern. Später (ab 21 s) bremst du zu spät, sodass die Geschwindigkeit abrupt von 198 → 96 km/h sinkt und das Fahrzeug bei Yaw‑Werten um –0,5 rad aus der idealen Linie gerät.
